# EDA RealWaste — Klasifikasi Jenis Sampah Rumah Tangga

Notebook ini mereproduksi seluruh bukti eksplorasi data pada laporan **Progres 2**:

- mengunduh dataset RealWaste (UCI id 908) secara otomatis,
- menjalankan `src/eda_realwaste.py`,
- menampilkan tabel dan gambar yang dipakai di laporan.

Dataset: 4.752 citra JPG 524 × 524 piksel, 9 kelas material, lisensi CC BY-NC-SA 4.0.
Belum ada pelatihan model pada tahap ini.

## 1. Unduh dan siapkan dataset

Arsip resmi berukuran 688,5 MB (SHA-256 `1ede08b3...895b2f3`). Server UCI tidak mendukung
*resume*, jadi unduhan akan diulang dari awal bila terputus. Di Colab, data disimpan di
`/content`; di lokal, di `data/` (tidak ikut repositori).

In [ ]:
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent

WORK = Path("/content") if Path("/content").is_dir() else REPO / "data"
WORK.mkdir(parents=True, exist_ok=True)
ZIP = WORK / "realwaste.zip"
RAW = WORK / "realwaste-main" / "RealWaste"
URL = "https://archive.ics.uci.edu/static/public/908/realwaste.zip"

if not RAW.exists():
    if not ZIP.exists():
        print("mengunduh", URL)
        urllib.request.urlretrieve(URL, ZIP)
    print("mengekstrak", ZIP)
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(WORK)

print("dataset siap:", RAW)
print("jumlah citra:", sum(1 for _ in RAW.rglob("*.jpg")))

## 2. Jalankan skrip EDA

Skrip yang sama dengan yang dilampirkan pada laporan. Argumen `--raw` menunjuk folder induk
kelas dan `--out` menunjuk folder keluaran.

In [ ]:
OUT = Path("/content/results") if Path("/content").is_dir() else REPO / "results"
script = REPO / "src" / "eda_realwaste.py"

hasil = subprocess.run(
    [sys.executable, str(script), "--raw", str(RAW), "--out", str(OUT)],
    capture_output=True, text=True)
print("exit code:", hasil.returncode)
print(hasil.stdout[-3000:])
print(hasil.stderr[-3000:])

## 3. Distribusi kelas dan ringkasan pemeriksaan

In [ ]:
import json

import pandas as pd

dist = pd.read_csv(OUT / "class_distribution.csv")
summary = json.loads((OUT / "summary.json").read_text(encoding="utf-8"))

ringkas = {k: summary[k] for k in (
    "total_citra", "jumlah_kelas", "ekstensi", "mode_warna", "resolusi_unik",
    "berkas_rusak", "duplikat_identik_grup", "near_duplicate_pasangan (dhash<=6)",
    "near_duplicate_lintas_kelas", "ukuran_byte", "brightness")}
print(json.dumps(ringkas, indent=2, ensure_ascii=False))
display(dist)

## 4. Kandidat kemiripan tinggi (dasar aturan anti-kebocoran)

In [ ]:
near = pd.read_csv(OUT / "near_duplicates.csv")
print("pasangan dHash <= 4:", int((near["jarak"] <= 4).sum()),
      "| rentang jarak:", near["jarak"].min(), "-", near["jarak"].max())
display(near[near["jarak"] <= 4].sort_values("jarak").head(15))

## 5. Visual EDA

In [ ]:
from IPython.display import Image, display

for f in sorted((OUT / "figures").glob("*.png")):
    print(f.name)
    display(Image(filename=str(f), width=760))

## 6. Temuan utama

1. **Kelas tidak seimbang**: Plastic 921 citra (19,38 %) berbanding Textile Trash 318 citra
   (6,69 %), rasio 2,90 : 1. Akurasi global saja tidak cukup; evaluasi memakai F1 makro dan
   confusion matrix.
2. **Tidak ada data rusak atau duplikat identik**, tetapi ada **25 pasangan mirip (dHash ≤ 4)**
   yang mencakup 17 klaster dan 41 citra, 8 di antaranya lintas kelas. Pasangan ini harus
   ditempatkan pada subset yang sama saat pembagian data (*group-aware split*).
3. **Pencahayaan relatif seragam** (median 157,98; hanya 18 citra < 120), tetapi **kontras
   berbeda antarkelas** (median 32,8 untuk Glass sampai 56,0 untuk Vegetation) sehingga objek
   transparan lebih sulit dipisahkan.
4. **Latar pengambilan serupa** karena seluruh citra berasal dari satu fasilitas; risiko model
   mempelajari latar belakang perlu diperiksa dengan Grad-CAM pada tahap evaluasi.

Rencana pembagian data: 70 % training (3.326), 15 % validasi (713), 15 % pengujian (713),
terstratifikasi dan group-aware dengan seed 42.